# `klygo.config` (`access.py`): Nested key access and mutation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/config/Document_Config_Access.ipynb)


Read, immutably assign, test existence (including explicit `None`), and delete nested keys using dot-separated paths.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare workspace

In [1]:
from tempfile import TemporaryDirectory
from klygo import config, files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-config-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-config-73qc0llg


## Jump to an API

[`get`](#get) · [`set`](#set) · [`has`](#has) · [`delete`](#delete)

<a id="get"></a>

### `config.get`

Return a nested configuration value by dot-separated path.

**Signature:** `config.get(config_data: dict[str, Any] | box.Box, key_path: str, default: Any = None) -> Any`

In [2]:
cfg = {"model": {"optimizer": {"lr": 0.001, "weight_decay": None}}}
print("Existing key:", config.get(cfg, "model.optimizer.lr"))
print("Missing key with fallback:", config.get(cfg, "model.optimizer.momentum", default=0.9))

Existing key: 0.001
Missing key with fallback: 0.9


In [3]:
# Error example: empty key path
try:
    config.get(cfg, "   ")
except ValueError as exc:
    print("ValueError ->", exc)

ValueError -> key_path must be a non-empty string


<a id="set"></a>

### `config.set`

Return a copied configuration with one nested value assigned.

**Signature:** `config.set(config_data: dict[str, Any] | box.Box, key_path: str, value: Any) -> box.Box`

In [4]:
updated = config.set(cfg, "model.optimizer.momentum", 0.937)
print("Updated copy:", updated.model.optimizer.to_dict())
print("Original unchanged:", "momentum" in cfg["model"]["optimizer"])

Updated copy: {'lr': 0.001, 'weight_decay': None, 'momentum': 0.937}
Original unchanged: False


In [5]:
# Error example: non-mapping input
try:
    config.set([1, 2], "model.batch", 16)
except TypeError as exc:
    print("TypeError ->", exc)

TypeError -> config_data must be Mapping, Box, got list


<a id="has"></a>

### `config.has`

Return whether every component of a nested key path exists (even when the value is `None`).

**Signature:** `config.has(config_data: dict[str, Any] | box.Box, key_path: str) -> bool`

In [6]:
print("has weight_decay (value is None):", config.has(cfg, "model.optimizer.weight_decay"))
print("has missing key:", config.has(cfg, "model.optimizer.nesterov"))

has weight_decay (value is None): True
has missing key: False


<a id="delete"></a>

### `config.delete`

Delete one nested key in place and report whether it existed.

**Signature:** `config.delete(config_data: MutableMapping[str, Any] | box.Box, key_path: str) -> bool`

In [7]:
mutable_cfg = {"model": {"batch": 16, "lr": 0.01}}
print("First delete:", config.delete(mutable_cfg, "model.lr"))
print("Second delete:", config.delete(mutable_cfg, "model.lr"))
print("Remaining:", mutable_cfg)

First delete: True
Second delete: False
Remaining: {'model': {'batch': 16}}


In [8]:
# Error example: non-mutable mapping
try:
    config.delete("not-a-mapping", "model.lr")
except TypeError as exc:
    print("TypeError ->", exc)

TypeError -> config_data must be MutableMapping, Box, got str


## Put it together

In [9]:
cfg = config.defaults({"model": {"name": "yolo", "lr": 0.01, "debug": True}})
if config.has(cfg, "model.debug"):
    config.delete(cfg, "model.debug")
cfg = config.set(cfg, "model.lr", 0.005)
print("Final lr:", config.get(cfg, "model.lr"), "| has debug:", config.has(cfg, "model.debug"))

Final lr: 0.005 | has debug: False


## Remember

- Dot-path helpers (`get`, `set`, `has`, `delete`) navigate nested mappings safely; `set()` returns a new mapping copy whereas `delete()` removes the key in place and returns a boolean.
- `has()` distinguishes a missing key (`False`) from a key whose value is explicitly `None` (`True`).
- Next in `klygo.config`: continue with [Document_Config_Inspection.ipynb](Document_Config_Inspection.ipynb) or return to [Document_Config_Overview.ipynb](Document_Config_Overview.ipynb).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.archive`](../archive/Document_Archive_Overview.ipynb).